<a href="https://colab.research.google.com/github/WaqasNoor08/Brand-Sentiment-Analysis-/blob/main/Brand_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brand Sentiment Analysis (YouTube Comments)

Khaadi **clothing** videos ke comments scrape karke sentiment analysis. Perfume, "Khadi" herbal/stationery aur IPO videos filter ho jati hain.

## Cell 1: Libraries install

In [ ]:
!pip install -q google-api-python-client vaderSentiment wordcloud

## Cell 2: Settings (yahan brand aur API key daalo)

In [ ]:
from getpass import getpass

BRAND = "Khaadi"             # <- brand ka naam (title filter bhi isi par chalta hai)
COMMENTS_PER_VIDEO = 500     # har video se max comments
MIN_VIDEO_COMMENTS = 5       # is se kam comments wali video skip
SEARCH_PAGES = 2             # har query ke pages (50 videos per page)
ORDER = "relevance"          # "viewCount" = bari videos = zyada comments
MODEL = "vader"              # "vader" ya "xlmr"

API_KEY = getpass("YouTube API key paste karo: ")

## Cell 3: Clothing videos dhoondo aur comments scrape karo

In [ ]:
import html, re, time
from datetime import date
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

yt = build("youtube", "v3", developerKey=API_KEY)

# Sirf clothing queries (pehle sirf "review" tha, is se perfume/IPO videos bhi aa jati thin)
queries = [f"{BRAND} sale", f"{BRAND} new collection 2026", f"{BRAND} lawn collection",
           f"{BRAND} Eid collection", f"{BRAND} pret collection", f"{BRAND} winter collection",
           f"{BRAND} haul", f"{BRAND} try on", f"{BRAND} unboxing", f"{BRAND} shopping vlog",
           f"{BRAND} 3 piece suit", f"{BRAND} flat 50% off"]

BRAND_RX = re.compile(r"khaadi|khaddi", re.I)           # title mein brand
CLOTHING_RX = re.compile(r"collection|pret|lawn|suit|dress|kurta|shalwar|kameez|3 ?pc|3 ?piece|2 ?piece|cut piece|"
                         r"sale|off|discount|haul|try.?on|unbox|eid|summer|winter|fabric|clothes|outfit|"
                         r"ready to wear|khaas|casual|signature|stitched|unstitched|trouser|dupatta|abaya|kids wear", re.I)
BLOCK_RX = re.compile(r"perfume|fragrance|attar|deo|body spray|shampoo|soap|hair|oil|henna|mehndi|herbal|ayurved|"
                      r"sketch|paper|saree|sari|meesho|\bipo\b|shares?\b|stock|psx|invest|go public|financials|valuation", re.I)

def is_clothing(title):
    t = html.unescape(title)
    return bool(BRAND_RX.search(t) and CLOTHING_RX.search(t) and not BLOCK_RX.search(t))

def search_videos():
    found = {}
    for q in queries:
        token = None
        for _ in range(SEARCH_PAGES):
            try:
                res = yt.search().list(q=q, part="id,snippet", type="video", maxResults=50,
                                       pageToken=token, order=ORDER, regionCode="PK").execute()
            except HttpError as e:
                print("search skip:", q, e); break
            for i in res.get("items", []):
                found[i["id"]["videoId"]] = html.unescape(i["snippet"]["title"])
            token = res.get("nextPageToken")
            if not token:
                break
        print(f"'{q}': {len(found)} candidates")
    return found

def video_stats(ids):
    out, ids = {}, list(ids)
    for i in range(0, len(ids), 50):
        res = yt.videos().list(part="statistics,snippet", id=",".join(ids[i:i + 50])).execute()
        for it in res.get("items", []):
            st = it.get("statistics", {})
            out[it["id"]] = dict(views=int(st.get("viewCount", 0)), video_likes=int(st.get("likeCount", 0)),
                                 video_comment_count=int(st.get("commentCount", 0)),
                                 video_published_at=it["snippet"]["publishedAt"],
                                 channel=it["snippet"]["channelTitle"])
    return out

def get_comments(video_id, title, max_comments):
    rows, token = [], None
    while len(rows) < max_comments:
        try:
            res = yt.commentThreads().list(part="snippet", videoId=video_id, maxResults=100,
                                           pageToken=token, textFormat="plainText",
                                           order="relevance").execute()
        except HttpError:
            break  # is video ke comments band hain
        for it in res["items"]:
            s = it["snippet"]["topLevelComment"]["snippet"]
            rows.append({"video_id": video_id, "video_title": title, "comment": s["textDisplay"],
                         "likes": s["likeCount"], "published_at": s["publishedAt"]})
        token = res.get("nextPageToken")
        if not token:
            break
        time.sleep(0.2)
    return rows[:max_comments]

cand = search_videos()
keep = {v: t for v, t in cand.items() if is_clothing(t)}
print(f"\n{len(cand)} candidates -> {len(keep)} {BRAND} clothing videos")
stats = video_stats(keep)
keep = {v: t for v, t in keep.items() if stats.get(v, {}).get("video_comment_count", 0) >= MIN_VIDEO_COMMENTS}

rows = []
for n, (vid, title) in enumerate(keep.items(), 1):
    rows += get_comments(vid, title, COMMENTS_PER_VIDEO)
    print(f"[{n}/{len(keep)}] {len(rows)} comments | {title[:50]}")

df = pd.DataFrame(rows).drop_duplicates(subset=["video_id", "comment"]).reset_index(drop=True)
df = df.merge(pd.DataFrame.from_dict(stats, orient="index").reset_index(names="video_id"), on="video_id", how="left")
df["video_url"] = "https://www.youtube.com/watch?v=" + df["video_id"]
df["collection_date"] = date.today().isoformat()
df["platform"] = "YouTube"
print("\nTotal comments:", len(df), "| Videos used:", df["video_id"].nunique())
df.head()

## Cell 4: Sentiment analysis

In [ ]:
import re

def clean(t):
    t = re.sub(r"http\S+|www\.\S+", " ", str(t))
    t = re.sub(r"@\w+", " ", t)
    return re.sub(r"\s+", " ", t).strip()

df["clean"] = df["comment"].map(clean)
df = df[df["clean"].str.len() > 2].reset_index(drop=True)

if MODEL == "vader":
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    sia = SentimentIntensityAnalyzer()
    scores = [sia.polarity_scores(t)["compound"] for t in df["clean"]]
    df["score"] = scores
    df["sentiment"] = ["Positive" if s >= 0.05 else "Negative" if s <= -0.05 else "Neutral"
                       for s in scores]
else:
    !pip install -q transformers torch
    from transformers import pipeline
    clf = pipeline("sentiment-analysis",
                   model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
                   truncation=True, max_length=128)
    res = clf(df["clean"].tolist(), batch_size=32)
    df["sentiment"] = [r["label"].capitalize() for r in res]
    df["score"] = [r["score"] for r in res]

counts = df["sentiment"].value_counts()
print(counts)
print((counts / len(df) * 100).round(1).astype(str) + " %")

## Cell 5: Charts

In [ ]:

import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

colors = {"Positive": "#2e9e5b", "Neutral": "#9aa0a6", "Negative": "#d64545"}

# 1) Overall distribution
counts.plot(kind="bar", color=[colors.get(k, "#555") for k in counts.index])
plt.title(f"{BRAND}: Sentiment Distribution"); plt.ylabel("Comments")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("distribution.png", dpi=150); plt.show()

# 2) Per video: sirf top 12 videos (zyada comments wali)
top = df["video_title"].value_counts().head(12).index
sub = df[df["video_title"].isin(top)]
pv = pd.crosstab(sub["video_title"].str[:35], sub["sentiment"], normalize="index") * 100
pv = pv[[c for c in ["Positive", "Neutral", "Negative"] if c in pv.columns]]
fig, ax = plt.subplots(figsize=(10, 6))
pv.plot(kind="barh", stacked=True, ax=ax, color=[colors[c] for c in pv.columns])
ax.set_title("Sentiment per Video (%), top 12 videos"); ax.set_ylabel("")
ax.legend(title="sentiment", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.savefig("per_video.png", dpi=150); plt.show()

# 3) Word clouds (faltu words hata kar)
extra = {"video", "videos", "please", "plz", "pls", "khadi", BRAND.lower(),
         "ka", "ki", "ke", "ko", "hai", "hy", "ho", "h", "ha", "ye", "yeh",
         "bhi", "hi", "se", "me", "mein", "kya", "kia", "ap", "aap", "api", "kr", "kar",
         "karo", "nhi", "ji", "sis", "sister", "mam", "madam", "apa", "bhai",
         "hain", "hn", "hun", "tha", "thi", "aur", "or", "wo", "woh", "sy", "py", "pe", "par",
         "liye", "lye", "jo", "koi", "ab", "tu", "tou", "to", "main", "mn", "ya", "na",
         "hum", "mujhe", "apki", "apka", "aapki", "aapka", "gi", "ga", "gy", "hy"}
stop = STOPWORDS | extra
for label in ["Positive", "Negative"]:
    text = " ".join(df.loc[df["sentiment"] == label, "clean"])
    if len(text) > 20:
        wc = WordCloud(width=900, height=450, background_color="white",
                       stopwords=stop, min_word_length=3).generate(text)
        plt.figure(figsize=(9, 4.5)); plt.imshow(wc); plt.axis("off")
        plt.title(f"{label} comments"); plt.savefig(f"wordcloud_{label.lower()}.png", dpi=150)
        plt.show()

## Cell 6: Sample comments dekho (accuracy check)

In [ ]:
for label in ["Positive", "Negative", "Neutral"]:
    print(f"\n=== {label} ===")
    for t in df[df["sentiment"] == label]["clean"].head(5):
        print("-", t[:150])

## Cell 7: Results download karo

In [ ]:
import glob, zipfile
from google.colab import files

df.to_csv("results.csv", index=False, encoding="utf-8-sig")
with zipfile.ZipFile("results.zip", "w") as z:
    for f in ["results.csv"] + glob.glob("*.png"):
        z.write(f)
files.download("results.zip")